In [1]:
import re
from docling.document_converter import DocumentConverter,PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
import os
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.do_cell_matching = False

KeyboardInterrupt: 

In [ ]:
def pdf2md_single(source) :
    converter = DocumentConverter(format_options={ InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options, backend=PyPdfiumDocumentBackend
            )
        }
    )
    result = converter.convert(source)
    return result.document.export_to_markdown()


#file = open('pdf2md.md','w',encoding='utf-8')
#file.write(pdf2md_single("../unique_sxolika_pdf/paper_16.pdf"))

In [ ]:
def paragraph_maker(text,maxpadding = 2) :
    lines = text.splitlines()
    paragraphs = []
    emptyline = 0
    paragraph = ''
    for i,line in enumerate(lines) :
        if i == len(lines) - 1 :
            paragraph = paragraph + line
            paragraphs.append(paragraph)
            continue
        if line == '' :
            emptyline = emptyline + 1
            if emptyline == maxpadding :
                if paragraph != '' :
                    paragraphs.append(paragraph)
                emptyline = 0
                paragraph = ''
        else :
            paragraph = paragraph + line + '\n'
            emptyline = 0
            if i == len(lines) - 1 :
                paragraphs.append(paragraph)
    return paragraphs

def paragraph_merger(paragraphs,min_par_size,threshold) :
    del_index = []
    newparagraphs = []
    for i,paragraph in enumerate(paragraphs) :
        if len(paragraph) < min_par_size :
            if paragraph != paragraphs[-1] :
                if len(paragraphs[i+1]) > threshold :
                    if not (paragraphs[i+1].startswith('##') or paragraphs[i+1].startswith('|') ) :
                        paragraphs[i+1] = paragraph + '\n MERGE \n' + paragraphs[i+1]
        else :
            newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_clean_image(paragraphs) :
    newparagraphs = []
    for paragraph in paragraphs :
        if paragraph.startswith('<!-- image -->') :
            continue
        else :
            newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_clean_dotlines(paragraphs) :
    newparagraphs = []
    for paragraph in paragraphs :
        if paragraph.startswith('.....') :
            continue
        else :
            newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_remove_artifacts(paragraphs) :
    newparagraphs = []
    for paragraph in paragraphs :
        if len(paragraph) < 4 :
            continue
        newparagraphs.append(paragraph)
    return newparagraphs

def paragraph_not_char_end(paragraph,chars) :
    #if not paragraph.startswith('##') :
    if not paragraph.endswith(chars) :
        paragraph = '[Out:Paragraph does not end with specified char]' + paragraph
    return paragraph

def all_paragraph_not_char_end(paragraphs,chars) :
    newparagraphs = []
    for paragraph in paragraphs :
        newparagraphs.append(paragraph_not_char_end(paragraph,chars))
    return newparagraphs

def paragraph_fix_broken_line(paragraphs) :
    ending_lower_pattern = re.compile(r'(.*[α-ωά-ώa-zΑ-ΩΆ-Ώ]\n)|(.*[0-9]\n)|(.*°C\n)|(.*\)\n)|(.*\-\n)')
    starting_lower_pattern = re.compile(r'([α-ωά-ώa-z])|([0-9])|(°)|\)')
    newparagraphs = []
    for i,paragraph in enumerate(paragraphs) :
        if re.match(ending_lower_pattern,paragraph) :
            if paragraph != paragraphs[-1] :
                if re.match(starting_lower_pattern,paragraphs[i+1]) :
                    paragraphs[i+1] = paragraph + 'MERGE\n' + paragraphs[i+1]
                    continue
        newparagraphs.append(paragraph)
    return newparagraphs

def print_text(paragraphs) :
    for paragraph in paragraphs :
        print(paragraph)
        print('\n\n\n')

def remove_ending_chunk(paragraphs) :
    paragraphs = paragraphs[::-1]
    for i,paragraph in enumerate(paragraphs) :
        if paragraph.startswith('##') :
            paragraphs[i] = '[Out:Paragraph is in ending chunk]' + paragraphs[i]
            return paragraphs[::-1]
        paragraphs[i] = '[Out:Paragraph is in ending chunk]' + paragraphs[i]
        #del paragraphs[-1]
    return paragraphs[::-1]

def remove_bibliography(paragraphs,bibliography) :
    newparagraphs = []
    bibliography_flag = False
    for paragraph in paragraphs :
        if paragraph.startswith(bibliography) or bibliography_flag == True :
            bibliography_flag = True
            paragraph = '[Out:Paragraph is Bibliography]' + paragraph
        if paragraph.startswith('##') :
            bibliography_flag = False
        newparagraphs.append(paragraph)
    return newparagraphs

def remove_glossary(paragraphs,glossary_tags) :
    newparagraphs = []
    bibliography_flag = False
    for paragraph in paragraphs :
        if paragraph.startswith(glossary_tags) or bibliography_flag == True :
            bibliography_flag = True
            paragraph = '[Out:Paragraph is Bibliography]' + paragraph
        if paragraph.startswith('##') :
            bibliography_flag = False
        newparagraphs.append(paragraph)
    return newparagraphs

def write_text(paragraphs,file) :
    text = '[Paragraph Begin]'
    for paragraph in paragraphs :
        text = text + paragraph + '[Paragraph End]\n\n\n[Paragraph Begin]'
    file.write(text) 

In [ ]:
os.makedirs('./test_output', exist_ok=True)
for i,file in enumerate(os.listdir('../md_output')) :
    if not file.endswith('.md'):
            continue
    try:
        with open('../md_output'+'/'+file, 'r', encoding='utf-8') as infile:
            text = infile.read()
    except Exception as e:
        print(f"Error reading {file}: {e}")
        continue

    endings = ('.\n','?\n','!\n',';\n','.\n',';\n',':\n','|\n')
    bibliography_tags = ('## ΞΕΝΟΓΛΩΣΣΗ','## ΕΛΛΗΝΙΚΗ','## Bl ΒΛΙΟΓΡΑΦΙΑ', '## ΒΛΙΟΓΡΑΦΙΑ')
    glossary_tags = ('## ΓΛΩΣΣΑΡΙ')

    paragraphs = paragraph_maker(text,maxpadding=1)
    paragraphs = paragraph_clean_image(paragraphs)
    paragraphs = paragraph_clean_dotlines(paragraphs)
    paragraphs = paragraph_remove_artifacts(paragraphs)
    paragraphs = paragraph_fix_broken_line(paragraphs)
    paragraphs = paragraph_merger(paragraphs,500,10)
    paragraphs = remove_bibliography(paragraphs,bibliography_tags)
    paragraphs = remove_glossary(paragraphs,glossary_tags)
    paragraphs = all_paragraph_not_char_end(paragraphs,endings)
    paragraphs = remove_ending_chunk(paragraphs)

    t_file = 'test_'+file

    output_file_path = os.path.join('./test_output', t_file)
    try:
        with open(output_file_path, 'w', encoding='utf-8') as output_file:
            write_text(paragraphs,output_file)
    except Exception as e:
            print(f"Error writing to {output_file_path}: {e}")
            continue